# 26_02 IsolationForest 이상탐지 개념코드

In [5]:
# [환경 설정] 한글 폰트 설정 및 필수 라이브러리 로드
# macOS, Windows, Linux, Google Colab 환경에 맞춰 한글 깨짐 없이 동작하도록 자동 감지 설정합니다.

import platform
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import pandas as pd
import numpy as np

try:
    import seaborn as sns
except ImportError:
    pass

# 운영체제(OS)별 한글 폰트 자동 설정
system_name = platform.system()
if system_name == 'Darwin':          # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
    plt.rcParams['font.sans-serif'] = ['AppleGothic', 'Apple SD Gothic Neo', 'NanumGothic', 'DejaVu Sans']
elif system_name == 'Windows':       # Windows
    plt.rcParams['font.family'] = 'Malgun Gothic'
    plt.rcParams['font.sans-serif'] = ['Malgun Gothic', 'NanumGothic', 'DejaVu Sans']
else:                               # Linux / Google Colab
    try:
        nanum_fonts = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name]
        if nanum_fonts:
            plt.rcParams['font.family'] = nanum_fonts[0]
        else:
            import subprocess
            subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], check=False, stdout=subprocess.DEVNULL)
            fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
            plt.rcParams['font.family'] = 'NanumGothic'
    except Exception:
        pass
    plt.rcParams['font.sans-serif'] = ['NanumGothic', 'DejaVu Sans']

# 마이너스 기호 깨짐 방지 및 Seaborn 폰트 동기화
plt.rcParams['axes.unicode_minus'] = False
try:
    if 'sns' in locals():
        sns.set_theme(style='whitegrid', font=plt.rcParams['font.family'])
except Exception:
    pass

print(f'✅ 환경 설정 완료! 현재 적용된 폰트: {plt.rcParams["font.family"]}')

Duplicate key in file WindowsPath('c:/Users/mzlap/Desktop/KNA-Data-analysis-1st/.venv/Lib/site-packages/matplotlib/mpl-data/matplotlibrc'), line 851 ('font.family : Malgun Gothic')
Duplicate key in file WindowsPath('c:/Users/mzlap/Desktop/KNA-Data-analysis-1st/.venv/Lib/site-packages/matplotlib/mpl-data/matplotlibrc'), line 852 ('axes.unicode_minus : False')


✅ 환경 설정 완료! 현재 적용된 폰트: ['Malgun Gothic']


새로운 패키지 설치가 필요합니다.

```bash
pip install scikit-learn
```

In [6]:
from sklearn.ensemble import IsolationForest

In [7]:
df = pd.read_csv("26_mimii_features.csv")

In [ ]:
# 만들기 → fit → predict 3단계
X = df[["rms", "spectral_centroid", "zero_crossing_rate"]]
# 레이블을 빼기위해 위처럼 사용


# 1. 만들기
iso = IsolationForest(contamination=0.18, random_state=42)  

# 학습 (보통 한 번만)
iso.fit(X)

# 예측
pred = iso.predict(X)
print(set(pred))

# 결과
# {np.int64(1), np.int64(-1)}

# 해석
# 1 : 정상 데이터(Inlier)
# -1 : 이상 데이터(Outlier)


{np.int64(1), np.int64(-1)}


In [ ]:
X = df[["rms", "spectral_centroid", "zero_crossing_rate"]]
iso = IsolationForest(contamination=0.18, random_state=42)
iso.fit(X)  # 모델링 학습
pred = iso.predict(X)
print(set(pred)) # -1과 1 두 값만 존재 (-1=이상, 1=정상)

# -1(이상)을 1로, 1(정상)을 0으로
pred_anom = (pred == -1).astype(int)
print(int(pred_anom.sum()))  # 40 개가 이상치를 알 수 있음


# 코드를 보면 데이터를 던져주고 기준을 너가 알아서 판단해서 이상치를 알려줘 라는 것임
# 함수 라이브러리를 통해 모델링 하여 앞으로 들어오는 이상값을 알 수 있도록 하는 것임
# 



{np.int64(1), np.int64(-1)}
40


In [ ]:
# 정답과 교차표 (채점)
print(pd.crosstab(df["label"], pred_anom).to_dict())  # 대조 기능 cosstab

# {0: {0: 166, 1: 14}, 1: {0: 14, 1: 26}}

# ==============================================================================
# [ 교차표(Crosstab) dict 출력 결과 해석 ]
# { pred_anom(예측값) : { df["label"](실제값) : 개수 } }
#
# { 0: {0: 166, 1: 14},  <-- 모델이 0(정상)으로 예측한 180개 중 [실제0: 166개, 실제1: 14개]
#   1: {0: 14,  1: 26} } <-- 모델이 1(이상치)로 예측한 40개 중 [실제0: 14개,  실제1: 26개]
# ==============================================================================
#
#                    [ 모델 예측 (pred_anom) ]
#                      0 (정상)     1 (이상치)     |  합계 (실제)
#  -----------------------------------------------+--------------
#  실제 (label) 0 |    166개          14개       |    180개  (실제 정상)
#               1 |     14개          26개       |     40개  (실제 이상치)
#  -----------------------------------------------+--------------
#     합계 (예측) |    180개          40개       |    220개  (전체 데이터)
#
# ==============================================================================
# [ 성능 요약 ]
# 1. 이상치 탐지율 (Recall): 26 / 40 = 65.0% (실제 이상치 40개 중 26개 적발)
# 2. 탐지 정확도 (Precision): 26 / 40 = 65.0% (이상치라 예측한 40개 중 실제 26개)
# 3. 전체 분류 정확도 (Accuracy): (166 + 26) / 220 = 87.27%
# ==============================================================================

{0: {0: 166, 1: 14}, 1: {0: 14, 1: 26}}


In [11]:
# 점수가 작은 순서가 곧 이상 우선순위
score = iso.score_samples(X)
print(round(float(score.min()), 3), round(float(score.max()), 3))
# -0.667 -0.372   (작을수록 이상)

-0.678 -0.372


In [12]:
# contamination을 바꾸면
# 비율을 키울수록 더 많이 잡힘
for c in [0.05, 0.10, 0.18, 0.25]:
    p = IsolationForest(contamination=c, random_state=42).fit_predict(X)
    n = int((p == -1).sum())
    real = int(((p == -1) & (df["label"] == 1)).sum())
    print(c, n, real)
# 출력: 0.05 11 10 / 0.10 22 16 / 0.18 40 27 / 0.25 55 35

0.05 11 10
0.1 22 14
0.18 40 26
0.25 55 35


In [14]:
# StandardScaler 전후 (트리 기반은 영향 작음)
# Xs = StandardScaler().fit_transform(X)
ps = IsolationForest(contamination=0.18, random_state=42).fit_predict(X)
real_s = int(((ps == -1) & (df["label"] == 1)).sum())
print(real_s)
# 출력: 27   (스케일링 전 27과 거의 같음)

26


In [ ]:
# 학습된 모델을 새 데이터에 적용 (예측만)
new = pd.read_csv("26_mimii_new.csv")
X_new = new[["rms", "spectral_centroid", "zero_crossing_rate"]]  # 같은 특징·순서
pred_new = iso.predict(X_new)        # 다시 학습하지 않고 predict만
print(int((pred_new == -1).sum()))
# 출력: 8   (label 없는 새 데이터에서 약 8개 이상)

8


# 실습

In [17]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

In [19]:
df = pd.read_csv("26_mimii_features.csv")
feats = ["rms", "spectral_centroid", "zero_crossing_rate"]

### 실습 1 — IsolationForest 첫 적용

In [19]:
X = df[feats]                                  # 특징만 (label 제외)
iso = IsolationForest(contamination=0.18, random_state=42)
iso.fit(X)
pred = iso.predict(X)
print(set(pred))

{np.int64(1), np.int64(-1)}


### 실습 2 — 이상 개수 세어보기 + 교차표

In [ ]:
df["pred_anom"] = (pred == -1).astype(int)     # -1(이상)→1, 1(정상)→0
print(int(df["pred_anom"].sum()))              # 40
print(pd.crosstab(df["label"], df["pred_anom"]))   # 실제 이상 40 중 27 맞힘

40
pred_anom    0   1
label             
0          166  14
1           14  26


### 실습 3 — 이상 점수 정렬해 Top N

In [ ]:
df["score"] = iso.score_samples(X)             # 작을수록 이상
top = df.sort_values("score").head(10)         # 가장 이상한 10개

# feats = ["rms", "spectral_centroid", "zero_crossing_rate"]
print(top[feats + ["score", "label"]])
print(int(top["label"].sum()))                 # 상위 10개 중 실제 이상 수

        rms  spectral_centroid  zero_crossing_rate     score  label
70   0.1599             1947.0              0.0891 -0.678462      1
54   0.0144             1864.1              0.1384 -0.638786      0
108  0.1467             2674.4              0.1037 -0.625262      1
157  0.1273             1577.0              0.0811 -0.613582      1
149  0.1303             2126.7              0.1419 -0.610869      1
81   0.0757             3055.9              0.1451 -0.608784      1
159  0.0670             2703.1              0.1722 -0.600776      1
23   0.1250             2846.4              0.0687 -0.598813      1
90   0.1259             2910.5              0.1078 -0.597624      1
218  0.1247             2783.3              0.0794 -0.583770      1
9


### 실습 4 — contamination 값 실험

In [ ]:
rows = []
for c in [0.05, 0.10, 0.18, 0.25]:
    p = IsolationForest(contamination=c, random_state=42).fit_predict(X)
    n = int((p == -1).sum())
    real = int(((p == -1) & (df["label"] == 1)).sum())  # 실제 이상치를 정확히 탐지한 개수 (True Positive / 적발 건수)
    rows.append({"contamination": c, "잡힌개수": n, "진짜이상": real})
print(pd.DataFrame(rows))                       # 11/22/40/55, 진짜 10/16/27/35

   contamination  잡힌개수  진짜이상
0           0.05    11    10
1           0.10    22    14
2           0.18    40    26
3           0.25    55    35


### 실습 5 — StandardScaler 적용 전후 비교

In [ ]:
real_raw = int(((iso.predict(X) == -1) & (df["label"] == 1)).sum())   # 약 27
# 원본 데이터(X)에서 실제 이상치를 정확히 적발한 건수 (TP)

Xs = StandardScaler().fit_transform(X)
# # 특성(Feature)별 단위를 통일하기 위한 표준화 처리 (Z-score normalization)
#   StandardScaler(): 데이터의 평균을 0, 표준편차를 1로 맞춰주는 스케일러 ($Z = \frac{X - \mu}{\sigma}$)
#   fit(): 데이터 X에서 각 열(컬럼)의 평균과 표준편차를 계산
#   transform(): 계산된 평균과 표준편차를 이용해 데이터 값들을 변환
#   fit_transform(): fit과 transform 과정을 한 번에 연속해서 수행

ps = IsolationForest(contamination=0.18, random_state=42).fit_predict(Xs)
# 표준화 데이터(Xs)로 Isolation Forest 모델 학습 및 이상치(-1) 예측
#   contamination=0.18: 전체 데이터 중 이상치 비율을 18%로 가정하여 판단 임계값 설정
#   random_state=42: 재현 가능성을 위해 난수 고정
#   fit_predict(Xs): 스케일링된 데이터 Xs로 모델 학습과 예측을 동시에 진행
#   real_scaled: 스케일링 전 원본 데이터 기준(real_raw)과 성능을 비교하기 위한 스케일링 후 탐지 성공 수

real_scaled = int(((ps == -1) & (df["label"] == 1)).sum())            # 약 27
# 스케일링 적용 데이터 기준 실제 이상치 탐지 성공 건수 (TP)

print(pd.DataFrame([{"구분": "원본", "진짜이상": real_raw},
                    {"구분": "스케일링", "진짜이상": real_scaled}]))

     구분  진짜이상
0    원본    26
1  스케일링    26


### 실습 6 — 새 데이터에 모델 적용

In [ ]:
new = pd.read_csv("26_mimii_new.csv")
# feats = ["rms", "spectral_centroid", "zero_crossing_rate"]

X_new = new[feats]                              # 학습 때와 같은 특징·순서
# X_new.head()
# 	rms	spectral_centroid	zero_crossing_rate
# 0	0.1104	2565.5	0.1073
# 1	0.0365	2098.5	0.1052
# 2	0.0550	1892.2	0.0954
# 3	0.0595	2047.4	0.0948
# 4	0.0367	2371.1	0.0910

pred_new = iso.predict(X_new)                   # 다시 학습하지 않고 예측만
# 새로운 데이터(X_new)의 이상치 여부 예측 (1: 정상, -1: 이상치)

print(int((pred_new == -1).sum()))              # 약 10
new["score"] = iso.score_samples(X_new)
print(new.sort_values("score").head(5)[feats + ["score"]])

8
       rms  spectral_centroid  zero_crossing_rate     score
13  0.1285             2974.6              0.1216 -0.633566
31  0.0629             3028.4              0.1203 -0.573328
6   0.1229             2668.5              0.0964 -0.557898
20  0.1286             1901.8              0.0926 -0.543498
0   0.1104             2565.5              0.1073 -0.526070


### 실습 7 — 두 방법 결과 교차 확인

In [33]:
# 1. 정상 데이터(label == 0)만 추출하여 Z-score의 기준점(평균, 표준편차)으로 설정
normal = df[df["label"] == 0]

# 2. Z-score 기반 단변량 이상치 탐지 (|Z| > 3.0)
z_any = pd.Series(False, index=df.index)
for f in feats:
    # 정상 데이터의 평균/표준편차 기준으로 규격화
    z = (df[f] - normal[f].mean()) / normal[f].std()
    # 단 하나의 피처라도 |Z| > 3.0 이면 이상치(True)로 처리
    z_any = z_any | (np.abs(z) > 3.0)

# 3. 탐지 결과를 0(정상), 1(이상치)로 변환하여 데이터프레임에 저장
df["z_anom"] = z_any.astype(int)  # 단변량(Z-score) 이상치 여부
df["if_anom"] = (
    pred_anom  # 다변량(Isolation Forest) 이상치 여부 (0: 정상, 1: 이상치)
)

# 4. 두 알고리즘 간 탐지 결과 교차표(Crosstab) 출력
print(pd.crosstab(df["z_anom"], df["if_anom"]))

# ==============================================================================
# [ 교차표(Crosstab) 결과 해석 ]
#
#               if_anom: 0 (IF 정상)   if_anom: 1 (IF 이상치)
# z_anom: 0           168개                    13개
# z_anom: 1            12개                    27개
#
# 1) [168개] (z=0, if=0): 둘 다 정상으로 판단한 '완전 정상 데이터'
# 2) [ 13개] (z=0, if=1): 단변량은 정상이나 조합이 기이한 '관계형(다변량) 이상치' (only_if)
# 3) [ 12개] (z=1, if=0): 단일 수치만 극단적이어서 Z-score만 걸려든 '단변량 전용 극단치'
# 4) [ 27개] (z=1, if=1): 두 모델 모두 이상치로 확정한 '명확한 공통 이상치'
#
# ※ 참고 (성능 평가 정의)
# - 오탐 (False Positive): 실제 정상(label=0)을 이상치(1)로 잘못 판단함
# - 미탐 (False Negative): 실제 이상치(label=1)를 정상(0)으로 놓침
# (위 교차표는 모델 간 비교표이므로, 오탐/미탐을 보려면 df['label']과 비교해야 함)
# ==============================================================================

# 5. Isolation Forest만 잡은 관계형(다변량) 이상치 후보 추출
only_if = df[(df["z_anom"] == 0) & (df["if_anom"] == 1)]
print("관계형(다변량) 이상치 후보 개수:", len(only_if))  # 출력값: 13


if_anom    0   1
z_anom          
0        168  13
1         12  27
관계형(다변량) 이상치 후보 개수: 13
